# Third Attempt Parallel code : ASRS

In [ ]:
!pip install --upgrade transformers tokenizers datasets safetensors --no-deps

In [ ]:
%%writefile benchmark.py
import os
# Set environment variables BEFORE importing torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import math
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer
from scipy import stats
from typing import Dict, List, Tuple

from accelerate import Accelerator


# =====================================================================
# 1. Activation Function Definitions
# =====================================================================

# class AdaptiveSmoothSwish_1(nn.Module):
#     """
#     Novel Activation Function: ASRS
#     Formula: f(x) = x * tanh(sigmoid(x)) + alpha * x * (1 - sigmoid(x))
#     """
#     def __init__(self, alpha_init: float = 0.01):
#         super(AdaptiveSmoothSwish, self).__init__()
#         self.alpha = nn.Parameter(torch.tensor(alpha_init, dtype=torch.float32))

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         sig_x = torch.sigmoid(x)
#         return x * torch.tanh(sig_x) + self.alpha * x * (1.0 - sig_x)

# class AdaptiveSmoothSwish_2(nn.Module):
#     """
#     Novel Activation Function: ASRS
#     Formula: f(x) = x * tanh(sigmoid(x)) + alpha * x * (1 - sigmoid(x))
    
#     Fixes Applied:
#     - Initialized alpha = 0.0 so initial forward pass matches baseline behavior.
#     - Clamped alpha in [-1.0, 1.0] to prevent runaway gradient dynamics.
#     """
#     def __init__(self, alpha_init: float = 0.0):
#         super(AdaptiveSmoothSwish, self).__init__()
#         self.alpha = nn.Parameter(torch.tensor(alpha_init, dtype=torch.float32))

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         sig_x = torch.sigmoid(x)
#         clamped_alpha = torch.clamp(self.alpha, min=-1.0, max=1.0)
#         return x * torch.tanh(sig_x) + clamped_alpha * x * (1.0 - sig_x)

# class AdaptiveSmoothSwish_3(nn.Module):
#     """
#     Novel Activation Function: Sigmoid-Gated ASRS
#     Formula: f(x) = x * tanh(sigmoid(x)) + sigmoid(alpha) * x * (1 - sigmoid(x))
#     """
#     def __init__(self, alpha_init: float = -4.0):
#         super(AdaptiveSmoothSwish, self).__init__()
#         self.alpha = nn.Parameter(torch.tensor(alpha_init, dtype=torch.float32))

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         sig_x = torch.sigmoid(x)
#         gated_alpha = torch.sigmoid(self.alpha)
#         return x * torch.tanh(sig_x) + gated_alpha * x * (1.0 - sig_x)


class Mish(nn.Module):
    """
    Mish Activation Function

    Formula:
        f(x) = x * tanh(softplus(x))
    """
    def __init__(self):
        super(Mish, self).__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.tanh(F.softplus(x))

# class AdaptiveSmoothSwish_4(nn.Module):
#     """
#     Novel Activation Function: ASRS
#     Formula: f(x) = x * tanh(sigmoid(x)) + alpha * x * (1 - sigmoid(x))
#     """
#     def __init__(self, alpha_init: float = 0.01):
#         super(AdaptiveSmoothSwish, self).__init__()
#         # self.alpha = nn.Parameter(torch.tensor(alpha_init, dtype=torch.float32))

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         sig_x = torch.sigmoid(x)
#         return x * torch.tanh(sig_x)

class ASGGELU(nn.Module):
    """
    Adaptive Symmetric Gated GELU (ASG-GELU)

    Formula:
        alpha = sigmoid(alpha_logit)
        gamma = softplus(gamma_raw) + eps

        gate = 1 + alpha * (2 * sigmoid(gamma * x) - 1)

        f(x) = GELU(x) * gate

    Only two scalar learnable parameters are introduced:
        alpha_logit
        gamma_raw
    """

    def __init__(
        self,
        init_alpha: float = 0.5,
        init_gamma: float = 1.0,
        eps: float = 1e-6,
        approximate: str = "none"
    ):
        super().__init__()

        self.eps = eps
        self.approximate = approximate

        # -------------------------------------------------------------
        # Initialize alpha in effective [0,1] space
        # alpha = sigmoid(alpha_logit)
        # -------------------------------------------------------------
        init_alpha = min(max(init_alpha, 1e-6), 1.0 - 1e-6)

        alpha_logit = math.log(
            init_alpha / (1.0 - init_alpha)
        )

        # -------------------------------------------------------------
        # Initialize gamma in positive space
        # gamma = softplus(gamma_raw) + eps
        # -------------------------------------------------------------
        init_gamma = max(init_gamma, eps)

        gamma_raw = math.log(
            math.exp(init_gamma) - 1.0
        )

        self.alpha_logit = nn.Parameter(
            torch.tensor(alpha_logit, dtype=torch.float32)
        )

        self.gamma_raw = nn.Parameter(
            torch.tensor(gamma_raw, dtype=torch.float32)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # Effective alpha in (0, 1)
        alpha = torch.sigmoid(self.alpha_logit)

        # Effective gamma > 0
        gamma = F.softplus(self.gamma_raw) + self.eps

        # Standard GELU
        gelu_x = F.gelu(
            x,
            approximate=self.approximate
        )

        # Symmetric adaptive gate
        gate = 1.0 + alpha * (
            2.0 * torch.sigmoid(gamma * x) - 1.0
        )

        return gelu_x * gate

    @torch.no_grad()
    def get_parameters(self):

        alpha = torch.sigmoid(self.alpha_logit)
        gamma = F.softplus(self.gamma_raw) + self.eps

        return {
            "alpha": alpha.item(),
            "gamma": gamma.item()
        }

# =====================================================================
# 2. Transformer Decoder Architecture
# =====================================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1)]


class CustomTransformerBlock(nn.Module):
    def __init__(self, d_model: int, nhead: int, dim_feedforward: int, act_class, dropout: float = 0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.act = act_class()
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.self_attn(src, src, src, attn_mask=attn_mask, is_causal=True)
        x = self.norm1(src + self.dropout(attn_out))
        ffn_out = self.linear2(self.act(self.linear1(x)))
        x = self.norm2(x + self.dropout(ffn_out))
        return x


class TransformerLM(nn.Module):
    def __init__(self, vocab_size: int, act_class, d_model: int = 256, nhead: int = 4,
                 dim_feedforward: int = 1024, num_layers: int = 4, max_len: int = 256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            CustomTransformerBlock(d_model, nhead, dim_feedforward, act_class)
            for _ in range(num_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def generate_square_subsequent_mask(self, sz: int, device: torch.device) -> torch.Tensor:
        return torch.triu(torch.ones(sz, sz, device=device) * float('-inf'), diagonal=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        mask = self.generate_square_subsequent_mask(seq_len, x.device)
        out = self.embedding(x) * math.sqrt(self.d_model)
        out = self.pos_encoder(out)
        for layer in self.layers:
            out = layer(out, mask)
        return self.fc_out(out)


# =====================================================================
# 3. Data Processing & Utilities
# =====================================================================

class TextDataset(Dataset):
    def __init__(self, data_tokens: List[int], seq_len: int):
        self.seq_len = seq_len
        tokens_tensor = torch.tensor(data_tokens, dtype=torch.long)
        num_chunks = (len(tokens_tensor) - 1) // seq_len
        self.inputs = tokens_tensor[:num_chunks * seq_len].view(-1, seq_len)
        self.targets = tokens_tensor[1 : num_chunks * seq_len + 1].view(-1, seq_len)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]


# Reduced per-GPU batch_size to 32 to safely fit T4 VRAM overhead during backward pass
def get_nlp_dataloaders(accelerator: Accelerator, seq_len: int = 128, batch_size: int = 32) -> Tuple[DataLoader, DataLoader, int]:
    with accelerator.main_process_first():
        raw_dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", trust_remote_code=True)
        tokenizer = AutoTokenizer.from_pretrained("gpt2")

        train_tokens = [tok for text in raw_dataset["train"]["text"] if text.strip() for tok in tokenizer.encode(text)]
        valid_tokens = [tok for text in raw_dataset["validation"]["text"] if text.strip() for tok in tokenizer.encode(text)]

    train_ds = TextDataset(train_tokens, seq_len)
    valid_ds = TextDataset(valid_tokens, seq_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    return train_loader, valid_loader, tokenizer.vocab_size


def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True


# =====================================================================
# 4. Training Engine
# =====================================================================

def train_and_eval_nlp(act_name: str, act_class, vocab_size: int, train_loader: DataLoader,
                       valid_loader: DataLoader, accelerator: Accelerator, epochs: int = 20) -> Tuple[float, float, float]: # Tuple[float, float, float, List[float]]:
    
    model = TransformerLM(vocab_size=vocab_size, act_class=act_class)

    act_params, network_params = [], []
    for name, param in model.named_parameters():
        if "alpha" in name:
            act_params.append(param)
        else:
            network_params.append(param)

    optimizer_groups = [{'params': network_params, 'weight_decay': 0.01, 'lr': 5e-4}]
    if act_params:
        optimizer_groups.append({'params': act_params, 'weight_decay': 0.0, 'lr': 5e-5})

    optimizer = optim.AdamW(optimizer_groups)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    model, optimizer, train_loader, valid_loader, scheduler = accelerator.prepare(
        model, optimizer, train_loader, valid_loader, scheduler
    )

    # alpha_history = []
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            with accelerator.accumulate(model):
                outputs = model(x)
                loss = criterion(outputs.view(-1, vocab_size), y.view(-1))
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

        scheduler.step()

        unwrapped_model = accelerator.unwrap_model(model)
        # if act_name == "Novel_ASRS":
        #     gated_alphas = [torch.sigmoid(p).item() for name, p in unwrapped_model.named_parameters() if "alpha" in name]
        #     alpha_history.append(float(np.mean(gated_alphas)))

    total_time = time.time() - start_time

    model.eval()
    total_val_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for x, y in valid_loader:
            outputs = model(x)
            loss = criterion(outputs.view(-1, vocab_size), y.view(-1))
            avg_loss = accelerator.gather_for_metrics(loss).mean()
            total_val_loss += avg_loss.item()
            total_batches += 1

    avg_val_loss = total_val_loss / total_batches
    perplexity = math.exp(avg_val_loss)

    # return perplexity, avg_val_loss, total_time, alpha_history
    return perplexity, avg_val_loss, total_time


# =====================================================================
# 5. Benchmarking Execution
# =====================================================================

def run_nlp_benchmark():
    # Increased gradient_accumulation_steps to 4 (Effective batch size = 32 * 2 GPUs * 4 = 256)
    accelerator = Accelerator(mixed_precision="fp16", gradient_accumulation_steps=4)
    
    if accelerator.is_main_process:
        print(f"Running Benchmark on {accelerator.num_processes} GPU(s) with FP16 Mixed Precision!\n")

    train_loader, valid_loader, vocab_size = get_nlp_dataloaders(accelerator)

    activations = {
        "ReLU": nn.ReLU,
        "GELU": nn.GELU,
        "SiLU (Swish)": nn.SiLU,
        "Mish" : Mish,
        # "Novel_ASRS": AdaptiveSmoothSwish,
        "ASGGELU" : ASGGELU
    }

    seeds = [42, 123, 999, 2024, 777]
    epochs = 20

    results = {name: {"ppl": [], "loss": [], "time": []} for name in activations}
    # asrs_alpha_trajectories = []

    for name, act_class in activations.items():
        if accelerator.is_main_process:
            print(f"\n--- Benchmarking Activation: {name} ({epochs} Epochs) ---")
        
        for seed in seeds:
            set_seed(seed)
            # ppl, loss, elapsed, alphas = train_and_eval_nlp(
            #     name, act_class, vocab_size, train_loader, valid_loader, accelerator, epochs=epochs
            # )

            ppl, loss, elapsed = train_and_eval_nlp(
                name, act_class, vocab_size, train_loader, valid_loader, accelerator, epochs=epochs
            )
            
            results[name]["ppl"].append(ppl)
            results[name]["loss"].append(loss)
            results[name]["time"].append(elapsed)

            if name == "Novel_ASRS":
                asrs_alpha_trajectories.append(alphas)

            if accelerator.is_main_process:
                print(f"  [Seed {seed:<4}] Val PPL: {ppl:.2f} | Val Loss: {loss:.4f} | Time: {elapsed:.1f}s")

            # Deep clean GPU cache after every run
            torch.cuda.empty_cache()

    if accelerator.is_main_process:
        print("\n" + "=" * 85)
        print(f"{'Activation':<15} | {'Val Perplexity (PPL) ↓':<25} | {'Val Loss ↓':<15} | {'Mean Time (s)':<12}")
        print("=" * 85)

        for name in activations:
            ppl_mean, ppl_std = np.mean(results[name]["ppl"]), np.std(results[name]["ppl"])
            loss_mean, loss_std = np.mean(results[name]["loss"]), np.std(results[name]["loss"])
            time_mean = np.mean(results[name]["time"])

            ppl_str = f"{ppl_mean:.2f} +/- {ppl_std:.2f}"
            loss_str = f"{loss_mean:.4f} +/- {loss_std:.4f}"

            print(f"{name:<15} | {ppl_str:<25} | {loss_str:<15} | {time_mean:<12.1f}")
        print("=" * 85)

        print("\n--- Statistical Significance Analysis (Welch's t-Test vs Novel_ASRS) ---")
        asrs_ppls = results["Novel_ASRS"]["ppl"]
        
        # for name in ["GELU", "SiLU (Swish)", "ReLU"]:
        #     baseline_ppls = results[name]["ppl"]
        #     t_stat, p_val = stats.ttest_ind(asrs_ppls, baseline_ppls, equal_var=False)
        #     is_better = np.mean(asrs_ppls) < np.mean(baseline_ppls)
        #     sig_str = f"Statistically Significant {'Improvement' if is_better else 'Regression'}" if p_val < 0.05 else "Not Significant"
        #     print(f"Novel_ASRS vs {name:<12}: p-value = {p_val:.4f} --> {sig_str}")

        # if asrs_alpha_trajectories:
        #     mean_alphas = np.mean(asrs_alpha_trajectories, axis=0)
        #     print(f"\nNovel_ASRS Effective Sigmoid(Alpha) Trajectory (Epoch 1 -> Epoch {epochs}):")
        #     print(f"  Initial: {mean_alphas[0]:.4f} | Mid: {mean_alphas[epochs//2]:.4f} | Final: {mean_alphas[-1]:.4f}")


if __name__ == "__main__":
    run_nlp_benchmark()

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/benchmark.py

===========================================
1. Experiment First with Loss Function 
===========================================
class AdaptiveSmoothSwish(nn.Module):
    """
    Novel Activation Function: ASRS
    Formula: f(x) = x * tanh(sigmoid(x)) + alpha * x * (1 - sigmoid(x))
    """
    def __init__(self, alpha_init: float = 0.01):
        super(AdaptiveSmoothSwish, self).__init__()
        self.alpha = nn.Parameter(torch.tensor(alpha_init, dtype=torch.float32))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sig_x = torch.sigmoid(x)
        return x * torch.tanh(sig_x) + self.alpha * x * (1.0 - sig_x)

=====================================================================================
Activation      | Val Perplexity (PPL) ↓    | Val Loss ↓      | Mean Time (s)
=====================================================================================
Novel_ASRS      | 291.90 +/- 0.53           | 5.6764 +/- 0.0018 | 594.9       
=====================================================================================

--- Statistical Significance Analysis (Welch's t-Test vs Novel_ASRS) ---

Novel_ASRS Effective Alpha Trajectory (Epoch 1 -> Epoch 20):
  Initial: 0.5025 | Mid: 0.5001 | Final: 0.4959
        
===========================================
2. Experiment Second with Loss Function 
===========================================
class AdaptiveSmoothSwish(nn.Module):
    """
    Novel Activation Function: ASRS
    Formula: f(x) = x * tanh(sigmoid(x)) + alpha * x * (1 - sigmoid(x))
    
    Fixes Applied:
    - Initialized alpha = 0.0 so initial forward pass matches baseline behavior.
    - Clamped alpha in [-1.0, 1.0] to prevent runaway gradient dynamics.
    """
    def __init__(self, alpha_init: float = 0.0):
        super(AdaptiveSmoothSwish, self).__init__()
        self.alpha = nn.Parameter(torch.tensor(alpha_init, dtype=torch.float32))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sig_x = torch.sigmoid(x)
        clamped_alpha = torch.clamp(self.alpha, min=-1.0, max=1.0)
        return x * torch.tanh(sig_x) + clamped_alpha * x * (1.0 - sig_x)

=====================================================================================
Activation      | Val Perplexity (PPL) ↓    | Val Loss ↓      | Mean Time (s)
=====================================================================================
Novel_ASRS      | 290.59 +/- 0.68           | 5.6719 +/- 0.0023 | 608.4       
=====================================================================================

--- Statistical Significance Analysis (Welch's t-Test vs Novel_ASRS) ---

Novel_ASRS Effective Sigmoid(Alpha) Trajectory (Epoch 1 -> Epoch 20):
  Initial: 0.5000 | Mid: 0.4976 | Final: 0.4935

===========================================
3. Experiment Third with Loss Function 
===========================================
class AdaptiveSmoothSwish(nn.Module):
    """
    Novel Activation Function: Sigmoid-Gated ASRS
    Formula: f(x) = x * tanh(sigmoid(x)) + sigmoid(alpha) * x * (1 - sigmoid(x))
    """
    def __init__(self, alpha_init: float = -4.0):
        super(AdaptiveSmoothSwish, self).__init__()
        self.alpha = nn.Parameter(torch.tensor(alpha_init, dtype=torch.float32))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sig_x = torch.sigmoid(x)
        gated_alpha = torch.sigmoid(self.alpha)
        return x * torch.tanh(sig_x) + gated_alpha * x * (1.0 - sig_x)
        
=====================================================================================
Activation      | Val Perplexity (PPL) ↓    | Val Loss ↓      | Mean Time (s)
=====================================================================================
ReLU            | 279.49 +/- 2.81           | 5.6329 +/- 0.0100 | 561.0       
GELU            | 269.99 +/- 1.04           | 5.5984 +/- 0.0039 | 562.7       
SiLU (Swish)    | 284.70 +/- 0.78           | 5.6514 +/- 0.0027 | 562.7       
Novel_ASRS      | 295.03 +/- 1.19           | 5.6871 +/- 0.0040 | 602.7       
=====================================================================================

--- Statistical Significance Analysis (Welch's t-Test vs Novel_ASRS) ---
Novel_ASRS vs GELU        : p-value = 0.0000 --> Statistically Significant Regression
Novel_ASRS vs SiLU (Swish): p-value = 0.0000 --> Statistically Significant Regression
Novel_ASRS vs ReLU        : p-value = 0.0001 --> Statistically Significant Regression

Novel_ASRS Effective Sigmoid(Alpha) Trajectory (Epoch 1 -> Epoch 20):
  Initial: 0.0180 | Mid: 0.0178 | Final: 0.0175

=====================================================================================
Activation      | Val Perplexity (PPL) ↓    | Val Loss ↓      | Mean Time (s)
=====================================================================================
ReLU            | 279.49 +/- 2.81           | 5.6329 +/- 0.0100 | 561.0       
GELU            | 269.99 +/- 1.04           | 5.5984 +/- 0.0039 | 562.7       
SiLU (Swish)    | 284.70 +/- 0.78           | 5.6514 +/- 0.0027 | 562.7     
Mish            | 281.42 +/- 0.75           | 5.6399 +/- 0.0027 | 596.9     
ASRS_1          | 291.90 +/- 0.53           | 5.6764 +/- 0.0018 | 594.9 
ASRS_2          | 290.59 +/- 0.68           | 5.6719 +/- 0.0023 | 608.4   
ASRS_3          | 295.03 +/- 1.19           | 5.6871 +/- 0.0040 | 602.7
ASRS_4          | 292.10 +/- 0.89           | 5.6771 +/- 0.0030 | 576.8 
=====================================================================================

=====================================================================================
Activation      | Val Perplexity (PPL) ↓    | Val Loss ↓      | Mean Time (s)
=====================================================================================
ReLU            | 279.49 +/- 2.81           | 5.6329 +/- 0.0100 | 561.0       
GELU            | 269.99 +/- 1.04           | 5.5984 +/- 0.0039 | 562.7       
SiLU (Swish)    | 284.70 +/- 0.78           | 5.6514 +/- 0.0027 | 562.7     
Mish            | 281.42 +/- 0.75           | 5.6399 +/- 0.0027 | 596.9     
ASGGELU         | 263.77 +/- 1.00           | 5.5751 +/- 0.0038 | 611.8
=====================================================================================